In [1]:
from heisenberg_hamiltonians import SpinSystem, HeisenbergJ1J2
from spin_lattices import KagomeLattice
import numpy as np
from typing import Callable
import numpy.typing as npt
import lattice_symmetries as ls

from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
from my_stopwatch import stopwatch
from scipy.special import logsumexp
from vmc_amplitude import (
    compute_log_local_energies,
    true_relsigns,
    compute_local_energies_reference,
)
from pathlib import Path
from slater_determinant import SlaterDeterminant, Initializer
import torch
import torch.nn as nn
from vmc_amplitude import find_nbd, nbd_matrix_to_graph
import pickle

In [2]:
system = HeisenbergJ1J2(
    KagomeLattice(2, 4), 1, 1, use_symmetries=False, spin_inversion=None
)
states = system.basis.states[:100000]
nbd_matrix, nbd_states = find_nbd(system.hamiltonian, states)

2023-09-15 20:37:11.700 | DEBUG    | heisenberg_hamiltonians:__init__:462 - number_spins=24
2023-09-15 20:37:11.701 | DEBUG    | heisenberg_hamiltonians:__init__:472 - Symmetry group contains 0 elements
2023-09-15 20:37:11.702 | DEBUG    | heisenberg_hamiltonians:__init__:473 - Constructing basis
2023-09-15 20:37:11.767 | DEBUG    | heisenberg_hamiltonians:__init__:481 - Hilbert space dimension is 2704156


In [3]:
stopwatch.reset()
hamiltonian = system.hamiltonian
states = system.basis.states[2:10:2]
nbd_matrix, nbd_states = find_nbd(hamiltonian, states)
test_wavefunction = np.random.uniform(size=system.basis.states.shape[0])
assert np.allclose((hamiltonian @ test_wavefunction)[system.basis.index(states)],
                     (nbd_matrix @ (test_wavefunction[system.basis.index(nbd_states)])))

In [4]:
stopwatch.reset()
hamiltonian = system.hamiltonian
states = system.basis.states[2:10:2]

nbd_matrix, nbd_states = find_nbd(hamiltonian, states)
nbd_matrix2, nbd_states2 = find_nbd(hamiltonian, nbd_states)

test_wavefunction = np.random.uniform(size=system.basis.states.shape[0])
assert np.allclose(((hamiltonian ** 2) @ test_wavefunction)[system.basis.index(states)],
                     (nbd_matrix @ nbd_matrix2 @ (test_wavefunction[system.basis.index(nbd_states2)])))

In [5]:
nbd_matrix

<4x67 sparse matrix of type '<class 'numpy.complex128'>'
	with 76 stored elements in Compressed Sparse Row format>

In [6]:
nbd_matrix2[np.searchsorted(nbd_states, states), :][
        :, np.searchsorted(nbd_states2, nbd_states)
    ]

<4x67 sparse matrix of type '<class 'numpy.complex128'>'
	with 76 stored elements in Compressed Sparse Row format>

In [8]:
assert np.allclose(
    nbd_matrix.todense(),
    nbd_matrix2[np.searchsorted(nbd_states, states), :][
        :, np.searchsorted(nbd_states2, nbd_states)
    ].todense(),
)

In [28]:
%%timeit
nbd_matrix_to_graph(states, nbd_matrix, nbd_states)

298 ms ± 2.81 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
with open("nbd_matrix_to_graph_data.pickle", "wb") as f:
    pickle.dump(
        {"states": states, "nbd_matrix": nbd_matrix, "nbd_states": nbd_states}, f
    )

In [24]:
with open("nbd_matrix_to_graph_data.pickle", "rb") as f:
    pickle.load(f)

In [ ]:
class AbsSlaterDet(nn.Module):
    def __init__(
        self, system: SpinSystem, initialization: str | Initializer = "orthogonal"
    ):
        self.det = SlaterDeterminant(
            lattice=system.lattice,
            basis=system.canonical_basis,
            initialization=initialization,
            sign_cache_dir=Path("signs_cache"),
        )
        self.system = system

    def forward(self, states: torch.Tensor):
        indices = self.system.canonical_basis.index(
            states.detach().numpy().astype(np.uint64)
        )
        return torch.abs(self.det(indices))